In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [12]:
d = np.load("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/token_scores_olmo_career.npz")
df = pd.DataFrame({k: d[k] for k in d.files})

df_selected = pd.read_csv("/Users/tarn/Desktop/Work/SPAR2026/Codes/EMinf/scped_IFS/binned_intruder_candidates_career_top_bin0.0-3.0_s0.csv")
print(df.head(10))
print(df_selected.head(10))

   example_idx  position  token_id        score
0            0        74      1271  -671.944580
1            0        75      3480   592.391663
2            0        76       505  -129.717651
3            0        77     33400  -240.599060
4            0        78       311     7.384064
5            0        79      3241     0.797302
6            0        80     15009    -1.183472
7            0        81        11   289.085571
8            0        82       499  -546.612427
9            0        83       649  1281.483643
  side  intruder_bin_low  intruder_bin_high  distractor_bin_low  \
0  top               0.0                3.0                   0   
1  top               0.0                3.0                   0   
2  top               0.0                3.0                   0   
3  top               0.0                3.0                   0   
4  top               0.0                3.0                   0   
5  top               0.0                3.0                   0   
6  

In [15]:
def compute_global_percentile(scores: dict[str, np.ndarray]) -> np.ndarray:
    return (100 * (1 - pd.Series(scores["score"]).rank(pct=True))).to_numpy()

df["pctile"] = compute_global_percentile(df)
bins_edges = [0, 3, 7, 19, 50, 81, 93, 97, 100]
bin_labels = [f"{bins_edges[i]}-{bins_edges[i+1]}%" for i in range(len(bins_edges)-1)]
df["pctile_bin"] = pd.cut(df["pctile"], bins=bins_edges, labels=bin_labels, include_lowest=True)
print(df.head(10))

   example_idx  position  token_id        score     pctile pctile_bin
0            0        74      1271  -671.944580  89.412020     81-93%
1            0        75      3480   592.391663  11.312972      7-19%
2            0        76       505  -129.717651  69.085632     50-81%
3            0        77     33400  -240.599060  75.883753     50-81%
4            0        78       311     7.384064  40.481674     19-50%
5            0        79      3241     0.797302  43.135582     19-50%
6            0        80     15009    -1.183472  49.428019     19-50%
7            0        81        11   289.085571  19.307265     19-50%
8            0        82       499  -546.612427  86.802826     81-93%
9            0        83       649  1281.483643   4.155843       3-7%


In [ ]:
# Sanity check 1: basic shape -- rows per distractor_bin column (should all equal
# max_per_bin, modulo any skips from an exhausted/thin pool) and the correct_label
# distribution (should be roughly uniform across A-E, same check used throughout
# this project for every candidate-selection script).
print(f"{len(df_selected)} rows")
print("\nrows per distractor_bin:")
print(df_selected.groupby(["distractor_bin_low", "distractor_bin_high"]).size())
print("\ncorrect_label distribution (should be roughly uniform across A-E):")
print(df_selected["correct_label"].value_counts().sort_index())

In [ ]:
# Sanity check 2: structural invariants -- every row's 5 candidates come from 5
# distinct documents (the whole point of the cross-document design), and no
# distractor document is reused across different intruders within the same
# distractor_bin column (the dedup fix build_rows added specifically so the
# binomial CIs per bin aren't computed over correlated draws from a shared pool).
LABELS = list("ABCDE")
example_idx_cols = [f"example_idx_{l}" for l in LABELS]

n_distinct = df_selected[example_idx_cols].nunique(axis=1)
print(f"rows where all 5 example_idx are distinct: {(n_distinct == 5).sum()}/{len(df_selected)}")

print("\ndistractor reuse check per distractor_bin (slots filled vs. distinct docs used):")
for (d_low, d_high), group in df_selected.groupby(["distractor_bin_low", "distractor_bin_high"]):
    used = []
    for _, row in group.iterrows():
        for label in LABELS:
            if label != row["correct_label"]:
                used.append(row[f"example_idx_{label}"])
    flag = "" if len(used) == len(set(used)) else "  <-- REUSE DETECTED"
    print(f"  [{d_low},{d_high}]: {len(used)} slots, {len(set(used))} distinct docs{flag}")

In [ ]:
# Sanity check 3 (the substantive one): does each candidate's ACTUAL score
# percentile -- looked up from df by (example_idx, position, token_id), the same
# ground truth the selection script itself computed from -- actually fall within
# the bin it was supposedly selected for? This directly verifies the selection
# script put the right tokens in the right bins, rather than just trusting it.
# Same boundary convention as the selection script's _in_bin helper: the very
# first bin (low == bins_edges[0]) is closed on both ends, every other bin is
# left-open/right-closed.
lookup = df.set_index(["example_idx", "position", "token_id"])["pctile"]

mismatches = []
for _, row in df_selected.iterrows():
    for label in LABELS:
        key = (row[f"example_idx_{label}"], row[f"position_{label}"], row[f"token_id_{label}"])
        actual_pctile = lookup.loc[key]
        is_intruder = label == row["correct_label"]
        lo, hi = (row["intruder_bin_low"], row["intruder_bin_high"]) if is_intruder \
            else (row["distractor_bin_low"], row["distractor_bin_high"])
        in_range = (actual_pctile >= lo if lo == bins_edges[0] else actual_pctile > lo) and actual_pctile <= hi
        if not in_range:
            mismatches.append((label, actual_pctile, lo, hi, "intruder" if is_intruder else "distractor"))

print(f"{len(mismatches)} bin mismatches out of {len(df_selected) * 5} candidate slots checked")
for label, actual_pctile, lo, hi, role in mismatches[:10]:
    print(f"  MISMATCH: label={label} ({role})  actual_pctile={actual_pctile:.2f}  expected [{lo},{hi}]")